In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter
from datetime import datetime

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

# Load env BEFORE importing ai_agents (its bootstrap validates required keys at import time).
from dotenv import load_dotenv
load_dotenv("/Users/utkarshumang/my_projects/lead-enricher-ai-be/.env")

from ai_agents.agents.email_finder import run_batch, run_single, SourceType, LeadStatus
from ai_agents.agents.email_finder.graph import build_graph
from google_utils.google_sheet import GoogleSheetService


# ── Config ──
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1MlzHZ-tk75blMDXUnHvgX4Z4WUXD6c1MEndqQxwu-sc/edit?usp=sharing"
SOURCE_SHEET = "Master_List_SAAS_more_5k"
OUTPUT_SHEET = "Email_Finder_more_5k"
SOURCE_TYPE = SourceType.OTHER          # generic SaaS / YouTube channels

# This list is YouTube channels, so enable the youtube_about_enricher node. For a
# channel with no website yet, it reads the channel About page and extracts the
# creator's website + socials, then the normal crawl/Perplexity flow finds the
# email. It uses a free HTTP GET first; ScrapingBee (SCRAPINGBEE_API_KEY) is only
# an optional fallback if YouTube blocks the request, so no plan is required.
YOUTUBE_LIST = True

CHECKPOINT_FILE = "email_finder_saas_more5k_checkpoint.json"
BATCH_SIZE = 50
CONCURRENCY = 5

# Only process rows whose email is still missing (Email Status none / captcha).
# Set to False to (re-)process every row, including the 784 that already have one.
SKIP_ROWS_WITH_EMAIL = True


In [ ]:
# ── v2 Graph structure (same agent as the podscan notebooks, + youtube enricher) ──
# canonical_builder
#   ├─ (has existing_email, no website) -> validate_existing_email
#   ├─ (has website) -> discover_urls -> crawl_page (fan-out) -> resolve_best_email
#   │     └─ (no candidates + FB link) -> fb_crawler -> perplexity_discovery
#   ├─ (youtube_list + channel URL, no website) -> youtube_about_enricher
#   │     ├─ (discovered website) -> discover_urls -> crawl_page -> resolve_best_email
#   │     └─ (no website) -> perplexity_discovery   (now with channel socials)
#   └─ (no website) -> perplexity_discovery
#
# For this SaaS list each row is a YouTube channel with only a channel URL, so the
# typical path is:
#   canonical_builder -> youtube_about_enricher -> url_discovery -> resolve_best_email
#   canonical_builder -> youtube_about_enricher -> perplexity_discovery   (no website found)

graph = build_graph()
print("Graph compiled OK")
print(f"Nodes: {list(graph.get_graph().nodes)}")
try:
    print(graph.get_graph().draw_ascii())
except Exception:
    print(graph.get_graph().draw_mermaid())


In [ ]:
def load_checkpoint(spreadsheet_id: str) -> dict:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            data = json.load(f)
            if data.get("spreadsheet_id") == spreadsheet_id:
                print(f"Resuming from checkpoint - {len(data['processed_ids'])} channels already processed")
                return data
    return {
        "spreadsheet_id": spreadsheet_id,
        "processed_ids": [],     # Channel IDs already run
        "found_count": 0,
        "not_found_count": 0,
        "failed_count": 0,
        "last_updated": None,
    }


def save_checkpoint(checkpoint: dict) -> None:
    checkpoint["last_updated"] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)


def clear_checkpoint() -> None:
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared")


In [ ]:
def build_saas_raw_row(row: dict) -> dict:
    """
    Convert a Master_List_SAAS row into a raw_row dict for the email finder.

    The canonical_builder LLM maps these keys -> CanonicalLead:
      Channel Name / Brand Name -> identity
      YouTube (Channel URL)      -> social_links.youtube
      Email                      -> existing_email (triggers validate path)
      Country / Topic            -> extra context for perplexity discovery
    """
    channel_name = (row.get("Channel Name") or "").strip()
    return {
        "Channel Name": channel_name,
        "Brand Name": channel_name,
        "YouTube": (row.get("Channel URL") or "").strip(),
        "Email": (row.get("Email") or "").strip(),
        "Country": (row.get("Country") or "").strip(),
        "Topic": (row.get("SubSheet") or "").strip(),
        "_channel_id": (row.get("Channel ID") or "").strip(),
        "_channel_url": (row.get("Channel URL") or "").strip(),
        "_existing_email": (row.get("Email") or "").strip(),
    }


def collect_saas_rows(
    sheet_service: GoogleSheetService,
    spreadsheet_id: str,
    already_processed: set[str],
) -> list[dict]:
    """Read SOURCE_SHEET, filter to rows needing an email, skip processed channels."""
    success, df = sheet_service.get_sheet_data(spreadsheet_id, SOURCE_SHEET)
    assert success, f"Failed to read sheet: {df}"

    stats = {"total": 0, "has_email_skipped": 0, "no_id": 0, "already_done": 0}
    raw_rows: list[dict] = []

    for _, row in df.iterrows():
        stats["total"] += 1
        rd = row.to_dict()
        existing_email = (rd.get("Email") or "").strip()

        if SKIP_ROWS_WITH_EMAIL and existing_email:
            stats["has_email_skipped"] += 1
            continue

        raw_row = build_saas_raw_row(rd)
        cid = raw_row["_channel_id"]
        if not cid:
            stats["no_id"] += 1
            continue
        if cid in already_processed:
            stats["already_done"] += 1
            continue
        raw_rows.append(raw_row)

    print(f"Collection summary for '{SOURCE_SHEET}':")
    print(f"  Total rows scanned:        {stats['total']}")
    print(f"  Skipped (already had email): {stats['has_email_skipped']}")
    print(f"  Skipped (no Channel ID):   {stats['no_id']}")
    print(f"  Skipped (already processed): {stats['already_done']}")
    print(f"  Channels to process:       {len(raw_rows)}")
    return raw_rows


def _format_email_source(best_email: dict) -> str:
    source = (best_email.get("source") or "").strip()
    note = (best_email.get("note") or "").strip()
    if source and note:
        return f"{source} - {note}"
    return note or source


def result_to_sheet_row(raw_row: dict, result: dict) -> list:
    best_email = result.get("best_email") or {}
    if isinstance(best_email, dict):
        email = best_email.get("email", "")
        source_display = _format_email_source(best_email)
        confidence = best_email.get("confidence", "")
    else:
        email, source_display, confidence = "", "", ""

    nodes = result.get("nodes_executed") or []
    errors = result.get("errors") or []

    return [
        raw_row.get("Channel Name", ""),
        raw_row.get("_channel_url", ""),
        raw_row.get("_existing_email", ""),
        email,
        source_display,
        str(confidence),
        result.get("status", ""),
        " -> ".join(nodes),
        "; ".join(errors) if errors else "",
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    ]


In [ ]:
def _create_sheet_tab(sheet_service: GoogleSheetService, spreadsheet_id: str, sheet_name: str) -> None:
    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    if success and sheet_name in sheet_names:
        return
    sheet_service.service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},
    ).execute()
    print(f"Created sheet tab '{sheet_name}'")


def _ensure_output_headers(sheet_service: GoogleSheetService, spreadsheet_id: str, sheet_name: str) -> None:
    _create_sheet_tab(sheet_service, spreadsheet_id, sheet_name)
    success, values = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A1:J1")
    if not success or not values:
        headers = [[
            "Channel Name",
            "Channel URL",
            "Existing Email",
            "Email Found",
            "Email Source",
            "Confidence",
            "Status",
            "Nodes Executed",
            "Errors",
            "Processed At",
        ]]
        sheet_service.append_rows(spreadsheet_id, sheet_name, headers)
        print(f"Added headers to {sheet_name}")


In [ ]:
def run_saas_email_finder(
    spreadsheet_id: str,
    limit: int | None = None,
    batch_size: int = BATCH_SIZE,
    concurrency: int = CONCURRENCY,
    dry_run: bool = False,
) -> dict:
    """
    Run the email finder agent over SOURCE_SHEET.

    Every processed channel (found or not) is written to OUTPUT_SHEET so the
    tab is a complete report. Resumable via CHECKPOINT_FILE (keyed by Channel ID).

    Args:
        limit:   process at most this many channels (use for a small test run)
        dry_run: if True, run the agent but do NOT write to the sheet
    """
    sheet_service = GoogleSheetService()
    checkpoint = load_checkpoint(spreadsheet_id)
    already_processed = set(checkpoint["processed_ids"])

    print(f"Scanning '{SOURCE_SHEET}'...")
    all_rows = collect_saas_rows(sheet_service, spreadsheet_id, already_processed)
    if limit is not None:
        all_rows = all_rows[:limit]
        print(f"  (limited to first {len(all_rows)} for this run)")

    if not all_rows:
        print("Nothing to process.")
        return checkpoint

    if not dry_run:
        _ensure_output_headers(sheet_service, spreadsheet_id, OUTPUT_SHEET)

    node_stats: Counter = Counter()
    source_stats: Counter = Counter()
    total_batches = (len(all_rows) + batch_size - 1) // batch_size

    for batch_start in range(0, len(all_rows), batch_size):
        batch_rows = all_rows[batch_start: batch_start + batch_size]
        batch_num = batch_start // batch_size + 1
        print(f"\nBatch {batch_num}/{total_batches} - {len(batch_rows)} channels")

        results = run_batch(
            rows=batch_rows,
            source_type=SOURCE_TYPE,
            concurrency=concurrency,
            youtube_list=YOUTUBE_LIST,
            cost_mode="high",   # full cascade incl. paid Perplexity fallback
        )

        out_rows = []
        out_ids = []
        for raw_row, result in zip(batch_rows, results):
            status = result.get("status", "")
            for node in (result.get("nodes_executed") or []):
                node_stats[node] += 1
            best = result.get("best_email") or {}
            if isinstance(best, dict) and best.get("source"):
                source_stats[best["source"].split(" - ")[0]] += 1

            if status == "email_found":
                checkpoint["found_count"] += 1
            elif status == "failed":
                checkpoint["failed_count"] += 1
            else:
                checkpoint["not_found_count"] += 1

            out_rows.append(result_to_sheet_row(raw_row, result))
            out_ids.append(raw_row["_channel_id"])

        if dry_run:
            print("  [dry_run] sample output rows:")
            for r in out_rows[:5]:
                print("   ", r[0], "|", r[3] or "(no email)", "|", r[6])
            checkpoint["processed_ids"].extend(out_ids)
            continue

        success, msg = sheet_service.append_rows(spreadsheet_id, OUTPUT_SHEET, out_rows)
        print(f"  -> Wrote {len(out_rows)} rows to {OUTPUT_SHEET}: {msg}")
        if success:
            checkpoint["processed_ids"].extend(out_ids)
            save_checkpoint(checkpoint)
            print(f"  -> Checkpoint saved | Found: {checkpoint['found_count']} | "
                  f"Not found: {checkpoint['not_found_count']} | Failed: {checkpoint['failed_count']}")
        else:
            print(f"  WARNING sheet write failed - batch will be retried next run")

    print(f"\n{'='*50}\nPipeline complete\n{'='*50}")
    print(f"  Found:     {checkpoint['found_count']}")
    print(f"  Not found: {checkpoint['not_found_count']}")
    print(f"  Failed:    {checkpoint['failed_count']}")
    if node_stats:
        print("\n  Node execution counts:")
        for node, count in node_stats.most_common():
            print(f"    {node}: {count}")
    if source_stats:
        print("\n  Email sources:")
        for source, count in source_stats.most_common():
            print(f"    {source}: {count}")
    return checkpoint


In [ ]:
# ── Preview: confirm access, columns, and how many channels need an email ──
sheet_service = GoogleSheetService()
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

ok, df = sheet_service.get_sheet_data(spreadsheet_id, SOURCE_SHEET)
assert ok, df
print(f"'{SOURCE_SHEET}': {len(df)} rows")
print(f"Columns: {list(df.columns)}")
missing = (df["Email"].astype(str).str.strip() == "").sum()
print(f"Rows missing an email: {missing}")
print(f"Rows that already have one: {len(df) - missing}")


In [ ]:
# ── Optional: single-channel smoke test (no sheet writes) ──
# Exercises the youtube_about_enricher path end-to-end (needs SCRAPINGBEE_API_KEY).
sample = build_saas_raw_row({
    "Channel Name": "Hasan Aboul Hasan",
    "Channel URL": "https://youtube.com/channel/UCl4nXWTkPOqmKlEmIN5_TJQ",
    "Email": "",
    "Country": "LB",
    "SubSheet": "leads_Saas_-_Micro_Saas",
    "Channel ID": "UCl4nXWTkPOqmKlEmIN5_TJQ",
})
result = run_single(sample, SOURCE_TYPE, youtube_list=YOUTUBE_LIST)
print("Status:", result.get("status"))
print("Nodes: ", " -> ".join(result.get("nodes_executed", [])))
print("Website discovered:", (result.get("lead") or {}).get("website"))
print("Email: ", result.get("best_email", {}))
print("Errors:", result.get("errors", []))


In [ ]:
# ── TEST RUN: first 10 channels, writes to the output tab ──
# Review the Email_Finder_more_5k tab afterwards, then run the full cell below.
test_result = run_saas_email_finder(
    spreadsheet_id=spreadsheet_id,
    limit=10,
    dry_run=False,
)


In [ ]:
# ── FULL RUN: all ~5,554 channels missing an email ──
# Resumable: re-run this cell after an interruption and it continues from the checkpoint.
# To start completely fresh, call clear_checkpoint() first.

# result = run_saas_email_finder(
#     spreadsheet_id=spreadsheet_id,
#     dry_run=False,
#     batch_size=BATCH_SIZE,
#     concurrency=CONCURRENCY,
# )
